In [1]:
!pip install openai-whisper
!pip install torch
!pip install transformers
!pip install nltk
!pip install reportlab
!pip install streamlit
!pip install pydub

In [2]:
!pip install openai-whisper torch transformers nltk reportlab streamlit pydub soundfile librosa imageio-ffmpeg

In [3]:
!pip install librosa soundfile

In [4]:
!pip install pydub

In [8]:
!pip uninstall transformers -y


Found existing installation: transformers 5.3.0
Uninstalling transformers-5.3.0:
  Successfully uninstalled transformers-5.3.0


In [9]:

!pip install transformers==4.36.2

  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
Using cached transformers-4.36.2-py3-none-any.whl (8.2 MB)
   ---------------------------------------- 0.0/566.4 kB ? eta -:--:--
   ------------------ --------------------- 262.1/566.4 kB ? eta -:--:--
   ---------------------------------------- 566.4/566.4 kB 2.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.2 MB 3.7 MB/s eta 0:00:01
   ----------------------- ---------------- 1.3/2.2 MB 3.4 MB/s eta 0:00:01
   --------------------------------- ------ 1.8/2.2 MB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 2.5 MB/s  0:00:00

  Attempting uninstall: huggingface-hub

    Found existing installation: huggingface_hub 1.7.2

   ---------------------------------------- 0/3 [huggingface-hub]
    Uninstalling huggingface_hub-1.7.2:
   ---------------------------------------- 0/3 [huggingface-hub]
      Succ

  You can safely remove it manually.


In [1]:
%%writefile transcription.py
# import whisper
# from pydub import AudioSegment
# import os

# # Load model (better accuracy than tiny)
# model = whisper.load_model("base")

# def convert_audio(input_path):
#     try:
#         if input_path.endswith(".aac") or input_path.endswith(".m4a"):
#             output_path = input_path + ".wav"
#             audio = AudioSegment.from_file(input_path)
#             audio.export(output_path, format="wav")
#             return output_path
#         return input_path
#     except Exception as e:
#         print("Conversion error:", e)
#         return input_path

# def transcribe_audio(audio_path):
#     try:
#         audio_path = convert_audio(audio_path)

#         # ✅ FIX: pass file path directly
#         result = model.transcribe(audio_path)

#         return result["text"]

#     except Exception as e:
#         return f"Error: {str(e)}"

# import whisper


# model = whisper.load_model("base")

# def transcribe_audio(audio_path):
#     try:
#         print("Processing file:", audio_path)

#         # 🔥 FORCE Whisper to handle everything
#         result = model.transcribe(audio_path)

#         print("Transcription success")

#         return result["text"]

#     except Exception as e:
#         print("TRANSCRIPTION ERROR:", e)
#         return f"Error: {str(e)}"

import whisper
import os
# load model once
model = whisper.load_model("base")

def transcribe_audio(audio_path):
    try:
        print("Processing:", audio_path)

        result = model.transcribe(audio_path)

        print("Done")

        return result["text"]

    except Exception as e:
        print("ERROR:", e)
        return f"Error: {str(e)}"

Overwriting transcription.py


In [2]:
%%writefile summarizer.py
# from transformers import pipeline
# import nltk

# try:
#     nltk.data.find('tokenizers/punkt')
# except LookupError:
#     nltk.download('punkt')

# from nltk.tokenize import sent_tokenize

# summarizer = pipeline(
#     "summarization",
#     model="sshleifer/distilbart-cnn-12-6"
# )

# def summarize_text(text):
#     chunks = [text[i:i+1000] for i in range(0, len(text), 1000)]

#     summary = ""
#     for chunk in chunks:
#         result = summarizer(chunk, max_length=100, min_length=30, do_sample=False)
#         summary += result[0]['summary_text'] + " "

#     return summary

# def extract_action_items(text):
#     sentences = sent_tokenize(text)
#     keywords = ["should", "must", "need", "todo", "action"]

#     actions = []
#     for s in sentences:
#         if any(word in s.lower() for word in keywords):
#             actions.append(s)

#     return actions

from transformers import pipeline
import nltk

# download punkt once
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

from nltk.tokenize import sent_tokenize

# 🔥 FIX: use supported task
summarizer = pipeline(
    "text2text-generation",
    model="sshleifer/distilbart-cnn-12-6"
)

def summarize_text(text):
    chunks = [text[i:i+1000] for i in range(0, len(text), 1000)]

    summary = ""
    for chunk in chunks:
        result = summarizer(chunk, max_length=100, min_length=30)
        summary += result[0]['generated_text'] + " "

    return summary

def extract_action_items(text):
    sentences = sent_tokenize(text)
    keywords = ["should", "must", "need", "todo", "action"]

    actions = []
    for s in sentences:
        if any(word in s.lower() for word in keywords):
            actions.append(s)

    return actions

Overwriting summarizer.py


In [3]:
# %%writefile summarizer.py
# from reportlab.lib.pagesizes import letter
# from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
# from reportlab.lib.styles import getSampleStyleSheet

# def generate_report(summary, actions, filename):

#     doc = SimpleDocTemplate(filename, pagesize=letter)
#     styles = getSampleStyleSheet()

#     content = []

#     content.append(Paragraph("AI Meeting Assistant Report", styles['Title']))
#     content.append(Spacer(1, 20))

#     content.append(Paragraph("Summary:", styles['Heading2']))
#     content.append(Spacer(1, 10))

#     for line in summary.split("."):
#         content.append(Paragraph(line.strip(), styles['Normal']))
#         content.append(Spacer(1, 8))

#     content.append(Spacer(1, 20))
#     content.append(Paragraph("Action Items:", styles['Heading2']))
#     content.append(Spacer(1, 10))

#     for action in actions:
#         content.append(Paragraph(action, styles['Normal']))
#         content.append(Spacer(1, 8))

#     doc.build(content)

In [4]:
%%writefile report_generator.py
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

def generate_report(summary, actions, filename):

    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()

    content = []

    content.append(Paragraph("AI Meeting Assistant Report", styles['Title']))
    content.append(Spacer(1, 20))

    content.append(Paragraph("Summary:", styles['Heading2']))
    content.append(Spacer(1, 10))

    for line in summary.split("."):
        content.append(Paragraph(line.strip(), styles['Normal']))
        content.append(Spacer(1, 8))

    content.append(Spacer(1, 20))
    content.append(Paragraph("Action Items:", styles['Heading2']))
    content.append(Spacer(1, 10))

    for action in actions:
        content.append(Paragraph(action, styles['Normal']))
        content.append(Spacer(1, 8))

    doc.build(content)

Overwriting report_generator.py


In [5]:
%%writefile email_sender.py
import smtplib
from email.message import EmailMessage

def send_email(receiver_email, file_path):

    sender_email = "your_email@gmail.com"
    sender_password = "your_app_password"  # 🔥 Use Gmail App Password

    msg = EmailMessage()
    msg['Subject'] = "Meeting Report"
    msg['From'] = sender_email
    msg['To'] = receiver_email
    msg.set_content("Your meeting report is attached.")

    with open(file_path, "rb") as f:
        file_data = f.read()

    msg.add_attachment(
        file_data,
        maintype="application",
        subtype="pdf",
        filename="meeting_report.pdf"
    )

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(sender_email, sender_password)
        smtp.send_message(msg)

Overwriting email_sender.py


In [6]:
%%writefile app.py
# import streamlit as st
# import os

# from transcription import transcribe_audio
# from summarizer import summarize_text, extract_action_items
# from report_generator import generate_report
# from email_sender import send_email

# st.set_page_config(page_title="AI Meeting Assistant")

# st.title("🤖 AI Smart Meeting Assistant")

# uploaded_file = st.file_uploader("Upload Meeting Audio", type=["wav","mp3","m4a","aac"])

# email = st.text_input("Enter customer email (optional)")

# if uploaded_file:

#     os.makedirs("uploads", exist_ok=True)
#     os.makedirs("reports", exist_ok=True)

#     file_path = os.path.join("uploads", uploaded_file.name)

#     with open(file_path, "wb") as f:
#         f.write(uploaded_file.getbuffer())

#     st.success("Audio uploaded successfully ✅")

#     if st.button("Process Meeting"):

#         with st.spinner("Processing... ⏳"):

#             transcript = transcribe_audio(file_path)

#             summary = summarize_text(transcript)

#             actions = extract_action_items(transcript)

#             report_path = "reports/meeting_report.pdf"

#             generate_report(summary, actions, report_path)

#         st.subheader("📄 Summary")
#         st.write(summary)

#         st.subheader("✅ Action Items")
#         st.write(actions)

#         with open(report_path, "rb") as f:
#             st.download_button(
#                 "⬇ Download Report",
#                 f,
#                 file_name="meeting_report.pdf"
#             )

#         if email:
#             try:
#                 send_email(email, report_path)
#                 st.success("📧 Report sent successfully")
#             except Exception as e:
#                 st.error(f"Email failed: {e}")

# import streamlit as st
# import os
# from pathlib import Path

# from transcription import transcribe_audio
# from summarizer import summarize_text, extract_action_items
# from report_generator import generate_report
# from email_sender import send_email

# # ------------------ CONFIG ------------------
# st.set_page_config(page_title="AI Meeting Assistant", layout="centered")

# st.title("🤖 AI Smart Meeting Assistant")

# # ------------------ INPUT ------------------
# uploaded_file = st.file_uploader(
#     "Upload Meeting Audio",
#     type=["wav", "mp3", "m4a", "aac"]
# )

# email = st.text_input("Enter customer email (optional)")

# # ------------------ MAIN LOGIC ------------------
# if uploaded_file is not None:

#     try:
#         # Create folders safely
#         upload_dir = Path("uploads")
#         report_dir = Path("reports")

#         upload_dir.mkdir(exist_ok=True)
#         report_dir.mkdir(exist_ok=True)

#         # Safe filename (important fix)
#         safe_filename = uploaded_file.name.replace(" ", "_")

#         file_path = upload_dir / safe_filename

#         # Save file
#         with open(file_path, "wb") as f:
#             f.write(uploaded_file.getbuffer())

#         st.success("✅ Audio uploaded successfully")

#         # Debug (important for your issue)
#         st.info(f"📂 File path: {file_path}")

#     except Exception as e:
#         st.error(f"❌ File upload failed: {e}")
#         st.stop()

#     # ------------------ PROCESS BUTTON ------------------
#     if st.button("🚀 Process Meeting"):

#         with st.spinner("Processing... ⏳"):

#             try:
#                 # ---------------- TRANSCRIPTION ----------------
#                 transcript = transcribe_audio(str(file_path))

#                 if not transcript or "Error" in transcript:
#                     st.error("❌ Transcription failed")
#                     st.stop()

#                 # ---------------- SUMMARY ----------------
#                 summary = summarize_text(transcript)

#                 # ---------------- ACTION ITEMS ----------------
#                 actions = extract_action_items(transcript)

#                 # ---------------- REPORT ----------------
#                 report_path = report_dir / "meeting_report.pdf"
#                 generate_report(summary, actions, str(report_path))

#             except Exception as e:
#                 st.error(f"❌ Processing error: {e}")
#                 st.stop()

#         # ------------------ OUTPUT ------------------
#         st.subheader("📄 Summary")
#         st.write(summary)

#         st.subheader("✅ Action Items")
#         st.write(actions)

#         # Download button
#         try:
#             with open(report_path, "rb") as f:
#                 st.download_button(
#                     "⬇ Download Report",
#                     f,
#                     file_name="meeting_report.pdf"
#                 )
#         except:
#             st.error("❌ Report file not found")

#         # ------------------ EMAIL ------------------
#         if email:
#             try:
#                 send_email(email, str(report_path))
#                 st.success("📧 Report sent successfully")
#             except Exception as e:
#                 st.error(f"❌ Email failed: {e}")

import streamlit as st
from pathlib import Path

from transcription import transcribe_audio
from summarizer import summarize_text, extract_action_items
from report_generator import generate_report
from email_sender import send_email

# ---------------- CONFIG ----------------
st.set_page_config(page_title="AI Meeting Assistant", layout="centered")
st.title("🤖 AI Smart Meeting Assistant")

# ---------------- INPUT ----------------
uploaded_file = st.file_uploader(
    "Upload Meeting Audio",
    type=["wav", "mp3", "m4a", "aac"]
)

email = st.text_input("Enter customer email (optional)")

# ---------------- MAIN ----------------
if uploaded_file is not None:

    upload_dir = Path("uploads")
    report_dir = Path("reports")

    upload_dir.mkdir(exist_ok=True)
    report_dir.mkdir(exist_ok=True)

    safe_filename = uploaded_file.name.replace(" ", "_")
    file_path = upload_dir / safe_filename

    with open(file_path, "wb") as f:
        f.write(uploaded_file.getbuffer())

    st.success("✅ Audio uploaded successfully")

    if st.button("🚀 Process Meeting"):

        with st.spinner("Processing... ⏳"):

            try:
                # 🔹 Transcription
                transcript = transcribe_audio(str(file_path))

                # if not transcript or "Error" in transcript:
                #     st.error("Transcription failed")
                #     st.stop()
                if not transcript or "Error" in transcript:
                      st.error(f"Transcription failed: {transcript}")
                      print("DEBUG ERROR:", transcript)
                      st.stop()

                # 🔹 Summary
                summary = summarize_text(transcript)

                # 🔹 Actions
                actions = extract_action_items(transcript)

                # 🔹 Report
                report_path = report_dir / "meeting_report.pdf"
                generate_report(summary, actions, str(report_path))

            except Exception as e:
                st.error(f"Processing error: {e}")
                st.stop()

        # ---------------- OUTPUT ----------------
        st.subheader("Summary")
        st.write(summary)

        st.subheader("Action Items")
        st.write(actions)

        # Download
        with open(report_path, "rb") as f:
            st.download_button(
                "⬇ Download Report",
                f,
                file_name="meeting_report.pdf"
            )

        # Email
        if email:
            try:
                send_email(email, str(report_path))
                st.success("Report sent successfully")
            except Exception as e:
                st.error(f"Email failed: {e}")

Overwriting app.py


In [ ]:
!streamlit run app.py